# Protótipo 2 — LSTM acionada por refeição com vetor contextual

**Pergunta:** quando aparece um registro de carboidratos, a glicemia recente + o contexto (carboidratos e bolus conhecidos naquele momento) permitem prever melhor a **mudança de glicemia em 30 e 60 min** do que a glicemia sozinha?

**Na prática:** a LSTM resume as 12 últimas glicemias em um vetor de 32 números. Ao lado dele concatenamos 5 números de contexto; uma camada final prevê `[G(t+30) − G(t), G(t+60) − G(t)]`. Treinamos duas redes (com e sem contexto) e comparamos com a persistência, **nas mesmas refeições**.

Este é um experimento retrospectivo: prevê o futuro observado, **não** o efeito causal de mudar uma dose. A variável `QUICK` substitui o `--quick` do script.

## 0. Configuração

Ajuste `DATA_DIR`. Requer PyTorch (já vem no Google Colab).

In [ ]:
import os, re, glob, json, time, platform, warnings, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

# ----------------------------- CONFIGURAÇÃO -----------------------------
QUICK = True                 # True = execução pequena (rápida); False = execução completa
SEED = 42
SUBJECTS_WANTED = [1, 2, 3]  # None = todos os participantes encontrados
DATA_DIR = "/content/drive/MyDrive/AZT1D 2025/CGM Records"   # <-- ajuste para a sua pasta

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_OUT = "/content/drive/MyDrive"
except ImportError:          # rodando fora do Colab
    BASE_OUT = "."
OUT_DIR = os.path.join(BASE_OUT, "results", "prototipo_02_refeicao")
os.makedirs(OUT_DIR, exist_ok=True)
print("QUICK =", QUICK, "| DATA_DIR =", DATA_DIR, "| OUT_DIR =", OUT_DIR)

## 1. Preparação dos dados

Mesmas regras do protótipo 1: grade de 5 min `(t−5, t]` rotulada no final (`ceil`), CGM sem interpolação (valores conflitantes → ausente), carboidratos/bolus somados por intervalo, zero = sem registro, `FoodDelivered` e `Basal` fora, glicemia inválida = ausente.

In [ ]:
def find_subject_files(data_dir):
    """Localiza os CSVs e identifica o participante pelo nome da pasta/arquivo (ex.: 'Subject 3')."""
    found = {}
    for f in glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True):
        m = re.search(r"Subject\s*_?(\d+)", f, flags=re.I)
        if m:
            found.setdefault(int(m.group(1)), f)
    return dict(sorted(found.items()))


def prepare_subject(path):
    """CSV bruto -> grade regular de 5 min com colunas cgm, carb, bolus. Devolve (grade, auditoria)."""
    audit = {}
    df = pd.read_csv(path)
    audit["linhas_brutas"] = len(df)

    # Alias ambíguo: sem coluna CGM não dá para garantir que os registros são CGM -> exclui
    if "CGM" not in df.columns:
        return None, {**audit, "excluido": "sem coluna 'CGM' (alias 'Readings (CGM / BGM)' não confirmado)"}

    n0 = len(df); df = df.drop_duplicates(); audit["duplicadas_removidas"] = n0 - len(df)
    df["EventDateTime"] = pd.to_datetime(df["EventDateTime"], errors="coerce")
    audit["datas_invalidas"] = int(df["EventDateTime"].isna().sum())
    df = df.dropna(subset=["EventDateTime"]).sort_values("EventDateTime")

    for c in ["CGM", "CarbSize", "TotalBolusInsulinDelivered"]:
        if c not in df.columns:
            df[c] = np.nan; audit[f"{c}_coluna_ausente"] = True
        raw = df[c]; num = pd.to_numeric(raw, errors="coerce")
        audit[f"{c}_nao_numerico"] = int((num.isna() & raw.notna()).sum())
        audit[f"{c}_ausente"] = int(raw.isna().sum())
        df[c] = num

    df.loc[df["CGM"] <= 0, "CGM"] = np.nan                       # glicemia inválida -> ausente (nunca zero)
    df.loc[df["CarbSize"] < 0, "CarbSize"] = np.nan
    df.loc[df["TotalBolusInsulinDelivered"] < 0, "TotalBolusInsulinDelivered"] = np.nan

    # Intervalo (t-5min, t] rotulado em t  ==  arredondar PARA CIMA para múltiplos de 5 min
    df["bin"] = df["EventDateTime"].dt.ceil("5min")
    g = df.groupby("bin")

    mm = g["CGM"].agg(["min", "max"])
    cgm = mm["min"].where(mm["min"] == mm["max"])                 # 1 único valor distinto -> mantém; conflito -> NaN
    audit["intervalos_cgm_conflitantes"] = int(((mm["min"] != mm["max"]) & mm["min"].notna()).sum())
    carb = g["CarbSize"].sum()                                    # soma dos registros distintos; NaN é ignorado
    bol = g["TotalBolusInsulinDelivered"].sum()

    idx = pd.date_range(df["bin"].min(), df["bin"].max(), freq="5min")
    grid = pd.DataFrame({"cgm": cgm.reindex(idx),
                         "carb": carb.reindex(idx).fillna(0.0),   # 0 = nenhum evento REGISTRADO
                         "bolus": bol.reindex(idx).fillna(0.0)}, index=idx)
    audit["passos_na_grade"] = len(grid)
    audit["passos_cgm_ausente"] = int(grid["cgm"].isna().sum())
    return grid, audit


def make_cuts(n):
    """60% treino / 20% validação / 20% teste, em ordem cronológica, como (início, fim) de índices da grade."""
    a, b = int(n * 0.6), int(n * 0.8)
    return {"treino": (0, a), "validacao": (a, b), "teste": (b, n)}


def valid_origins(cgm, part, back_cgm, back_ev, fwd):
    """Índices de origem t válidos dentro de UMA partição.
    - CGM sem nenhuma lacuna de t-back_cgm até t+fwd (segmento contínuo até o alvo);
    - todo o suporte (inclusive a janela de eventos back_ev e o alvo) dentro da partição."""
    s, e = part
    ok = ~np.isnan(cgm)
    cs = np.concatenate([[0], np.cumsum(ok)])
    i = np.arange(s + max(back_cgm, back_ev), e - fwd)
    lo, hi = i - back_cgm, i + fwd
    return i[(cs[hi + 1] - cs[lo]) == (hi - lo + 1)]


def prefix(a):
    """Soma acumulada com zero na frente: soma dos passos a..b = p[b+1] - p[a]."""
    return np.concatenate([[0.0], np.cumsum(a)])


def first_run(idx, min_len):
    """Primeiro trecho de origens consecutivas (passo 1 da grade) com pelo menos min_len pontos."""
    if len(idx) == 0: return None
    breaks = np.where(np.diff(idx) != 1)[0] + 1
    for run in np.split(np.arange(len(idx)), breaks):
        if len(run) >= min_len: return run
    return None


def md_table(df, nd=2):
    cols = list(df.columns)
    L = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        L.append("| " + " | ".join(f"{v:.{nd}f}" if isinstance(v, (float, np.floating)) else str(v) for v in r) + " |")
    return "\n".join(L)


def mae_rmse(y, p):
    if len(y) == 0: return np.nan, np.nan
    e = np.asarray(p) - np.asarray(y)
    return float(np.mean(np.abs(e))), float(np.sqrt(np.mean(e ** 2)))

In [ ]:
files = find_subject_files(DATA_DIR)
print("Participantes encontrados:", list(files))
if SUBJECTS_WANTED is not None:
    files = {k: v for k, v in files.items() if k in SUBJECTS_WANTED}
assert files, "Nenhum CSV encontrado: confira DATA_DIR (e se o Drive foi montado)."

t0 = time.time()
grids, audits = {}, {}
for sid, path in files.items():
    g, a = prepare_subject(path)
    audits[sid] = a
    if g is not None: grids[sid] = g
t_prep_load = time.time() - t0

pd.DataFrame(audits).T

## 2. Divisão temporal

60/20/20 cronológico por participante. Os exemplos serão gerados **depois**, exigindo que histórico e ambos os alvos fiquem na mesma partição. Refeições nunca são divididas aleatoriamente entre treino e teste.

In [ ]:
cuts, cut_times = {}, {}
for sid, g in grids.items():
    cuts[sid] = make_cuts(len(g))
    cut_times[sid] = {p: [str(g.index[s]), str(g.index[e - 1])] for p, (s, e) in cuts[sid].items()}
pd.DataFrame({sid: {p: f"{v[0]} -> {v[1]}" for p, v in d.items()} for sid, d in cut_times.items()}).T

## 3. Disparo e exemplos

**Disparo:** todo intervalo de 5 min com `CarbSize > 0`; a origem *t* é o **fechamento** desse intervalo. A resolução é de 5 min — não afirmamos saber o instante exato da ingestão. Dois intervalos consecutivos com carboidrato geram dois exemplos (regra simples, documentada).

Para cada origem (índice `i`):

| Saída | Conteúdo | Passos da grade |
|---|---|---|
| `X_cgm` (12×1) | `G(t−55) … G(t)` | `cgm[i-11 … i]` |
| contexto 1 | carboidratos do intervalo atual | `carb[i]` |
| contexto 2 | bolus do intervalo atual | `bol[i]` |
| contexto 3 | carboidratos em `(t−60, t−5]` | passos `i-11 … i-1` |
| contexto 4 | bolus em `(t−60, t−5]` | passos `i-11 … i-1` |
| contexto 5 | bolus em `(t−120, t−60]` | passos `i-23 … i-12` |
| `y` | `[G(t+30)−G(t), G(t+60)−G(t)]` | `cgm[i+6]`, `cgm[i+12]` |

Nada posterior à origem entra no contexto (nenhum bolus futuro). Aceitamos só exemplos com histórico completo e **CGM contínua até +60 min**, mas **não** excluímos refeições seguidas de outros bolus/refeições: o alvo é o futuro observado, incluindo intervenções desconhecidas na origem (limitação a documentar).

In [ ]:
BACK_CGM, BACK_EV, FWD = 11, 23, 12
CTX_NAMES = ["carb_atual", "bolus_atual", "carb_ant_60", "bolus_ant_60", "bolus_ant_120"]


def build_p2(sid, grid, cuts_sid):
    cgm = grid["cgm"].values
    carb, bol = grid["carb"].values, grid["bolus"].values
    cc, bc = prefix(carb), prefix(bol)
    out = {}
    for part, rng in cuts_sid.items():
        cand = valid_origins(cgm, rng, BACK_CGM, BACK_EV, FWD)
        i = cand[carb[cand] > 0]                                       # disparo: CarbSize > 0 no intervalo
        if len(i) == 0: continue
        X = np.stack([cgm[i - k] for k in range(11, -1, -1)], axis=1)   # (n, 12)
        C_ = np.stack([carb[i], bol[i],
                       cc[i] - cc[i - 11],            # carboidratos i-11 ... i-1
                       bc[i] - bc[i - 11],            # bolus       i-11 ... i-1
                       bc[i - 11] - bc[i - 23]], 1)   # bolus       i-23 ... i-12
        Y = np.stack([cgm[i + 6] - cgm[i], cgm[i + 12] - cgm[i]], 1)
        meta = pd.DataFrame({"subject": sid, "origin_idx": i, "t": grid.index[i], "g0": cgm[i]})
        out[part] = (X, C_, Y, meta)
    return out


built, discarded = {}, {}
for sid, g in grids.items():
    r = build_p2(sid, g, cuts[sid])
    if set(r) != {"treino", "validacao", "teste"}:
        discarded[sid] = f"refeições insuficientes (partições com exemplos: {sorted(r)})"
    else:
        built[sid] = r
eligible = sorted(built)
if QUICK: eligible = eligible[:3]
print("Elegíveis:", eligible, "| excluídos:", discarded)

D = {}
for part in ["treino", "validacao", "teste"]:
    X = np.concatenate([built[s][part][0] for s in eligible])
    C_ = np.concatenate([built[s][part][1] for s in eligible])
    Y = np.concatenate([built[s][part][2] for s in eligible])
    meta = pd.concat([built[s][part][3] for s in eligible], ignore_index=True)
    D[part] = (X, C_, Y, meta)
counts = pd.DataFrame({p: {f"Subject {s}": len(built[s][p][0]) for s in eligible} for p in D})
counts.loc["TOTAL"] = counts.sum()
counts

In [ ]:
# Bloqueio explícito: sem refeições nas três partições não há como treinar (não trocamos por divisão aleatória)
assert all(len(D[p][0]) > 0 for p in D), "Refeições insuficientes para treino/validação/teste: veja a tabela acima."
# Verificação: bolus/carbo do contexto só vêm de passos <= i (por construção) e alvos estão a +6 e +12 passos
for s in eligible:
    for part, (s0, e0) in cuts[s].items():
        if part in built[s]:
            idx = built[s][part][3]["origin_idx"]
            assert idx.min() - BACK_EV >= s0 and idx.max() + FWD <= e0 - 1
print("Verificações de partição OK")

## 4. Escalas (ajustadas só no treino)

Três escalas: uma para toda a CGM (média/desvio escalares), uma para as 5 variáveis de contexto e uma para as 2 saídas. Depois do modelo, **desfazemos** a escala das saídas e somamos `G(t)` para voltar a mg/dL antes de calcular métricas.

In [ ]:
Xtr, Ctr, Ytr, _ = D["treino"]
cgm_mu, cgm_sd = float(Xtr.mean()), float(Xtr.std())
ctx_sc = StandardScaler().fit(Ctr)       # colunas constantes ficam com escala 1 (sem divisão por zero)
y_sc = StandardScaler().fit(Ytr)


def to_tensors(part):
    X, C_, Y, _ = D[part]
    x = torch.tensor(((X - cgm_mu) / cgm_sd)[..., None], dtype=torch.float32)   # (n, 12, 1)
    c = torch.tensor(ctx_sc.transform(C_), dtype=torch.float32)                 # (n, 5)
    y = torch.tensor(y_sc.transform(Y), dtype=torch.float32)                    # (n, 2)
    return x, c, y

T = {p: to_tensors(p) for p in D}
print({p: tuple(t.shape) for p, t in zip(D, [T[p][0] for p in D])})

## 5. Arquitetura

```text
CGM (12×1) → LSTM(32) → último estado (32)
com contexto:   concat(32 + 5) = 37 → Linear(37,16) → ReLU → Linear(16,2)
sem contexto:   32            → Linear(32,16) → ReLU → Linear(16,2)
```

A rede "só CGM" é **a mesma** com a camada final recebendo apenas os 32 números da LSTM; a diferença entre as duas é só o contexto, então a comparação mede o **ganho incremental** dele.

Treino: MSE nas saídas padronizadas, Adam `lr=0.001`, lote 64, semente 42, até 15 épocas (3 no modo rápido), parada após 3 épocas sem melhora da validação, restaurando o melhor estado.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


class Net(nn.Module):
    def __init__(self, use_ctx, n_ctx=5):
        super().__init__()
        self.use_ctx = use_ctx
        self.lstm = nn.LSTM(1, 32, batch_first=True)
        self.head = nn.Sequential(nn.Linear(32 + (n_ctx if use_ctx else 0), 16), nn.ReLU(), nn.Linear(16, 2))

    def forward(self, x, c):
        _, (h, _) = self.lstm(x)
        h = h[-1]                                   # último estado oculto (n, 32)
        if self.use_ctx: h = torch.cat([h, c], dim=1)
        return self.head(h)


@torch.no_grad()
def predict_scaled(net, part, bs=1024):
    net.eval(); x, c, _ = T[part]
    return torch.cat([net(x[k:k+bs].to(device), c[k:k+bs].to(device)).cpu() for k in range(0, len(x), bs)]).numpy()


def val_loss(net):
    p = torch.tensor(predict_scaled(net, "validacao")); return float(nn.functional.mse_loss(p, T["validacao"][2]))


def train_net(use_ctx, max_epochs, patience=3):
    torch.manual_seed(SEED); np.random.seed(SEED)
    net = Net(use_ctx).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    gen = torch.Generator().manual_seed(SEED)
    x, c, y = T["treino"]; n = len(x)
    best, best_state, bad, hist = np.inf, None, 0, []
    for ep in range(max_epochs):
        net.train(); perm = torch.randperm(n, generator=gen)
        for k in range(0, n, 64):
            b = perm[k:k + 64]
            opt.zero_grad()
            loss = nn.functional.mse_loss(net(x[b].to(device), c[b].to(device)), y[b].to(device))
            loss.backward(); opt.step()
        v = val_loss(net); hist.append(v)
        print(f"  época {ep+1}: perda de validação = {v:.4f}")
        if v < best - 1e-9:
            best, bad = v, 0
            best_state = {k: t.detach().cpu().clone() for k, t in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(best_state)                 # restaura o melhor estado de validação
    return net, hist

MAX_EPOCHS = 3 if QUICK else 15
t0 = time.time()
print("LSTM só CGM"); net_cgm, h_cgm = train_net(False, MAX_EPOCHS)
print("LSTM CGM + contexto"); net_ctx, h_ctx = train_net(True, MAX_EPOCHS)
t_train = time.time() - t0

## 6. Previsões em mg/dL

`delta_previsto = inverse_transform(saída)`; `G_previsto = G(t) + delta_previsto`. A persistência prevê delta zero nos dois horizontes.

In [ ]:
t0 = time.time()
frames = []
for part in D:
    X, C_, Y, meta = D[part]
    d = meta.copy(); d["particao"] = part
    for j, h in enumerate([30, 60]):
        d[f"real_{h}"] = d["g0"] + Y[:, j]
    d["ctx"] = [json.dumps(dict(zip(CTX_NAMES, np.round(r, 2).tolist()))) for r in C_]
    for name, net in [("lstm_cgm", net_cgm), ("lstm_cgm_ctx", net_ctx)]:
        dl = y_sc.inverse_transform(predict_scaled(net, part))
        for j, h in enumerate([30, 60]):
            d[f"{name}_{h}"] = d["g0"] + dl[:, j]
    for h in [30, 60]:
        d[f"persistencia_{h}"] = d["g0"]
    frames.append(d)
pred = pd.concat(frames, ignore_index=True)
t_infer = time.time() - t0

# mesmas refeições para todos os previsores: todas as colunas existem para todas as linhas
cols_pred = [c for c in pred.columns if c.startswith(("persistencia_", "lstm_"))]
assert pred[cols_pred].notna().all().all()
print(f"treino: {t_train:.1f}s | inferência: {t_infer:.1f}s")

## 7. Métricas (MAE/RMSE em mg/dL)

Por horizonte (30 e 60 min), por partição, global e por participante. Atenção: a população aqui são **refeições** — não compare diretamente com as métricas do protótipo 1 (que avalia todos os instantes).

In [ ]:
rows = []
for part in ["validacao", "teste"]:
    d = pred[pred.particao == part]
    for h in [30, 60]:
        for who, sub in [("AGREGADO", d)] + [(f"Subject {s}", d[d.subject == s]) for s in eligible]:
            for m in ["persistencia", "lstm_cgm", "lstm_cgm_ctx"]:
                mae, rmse = mae_rmse(sub[f"real_{h}"], sub[f"{m}_{h}"])
                rows.append(dict(particao=part, modelo=m, horizonte_min=h, participante=who,
                                 n_refeicoes=len(sub), MAE=mae, RMSE=rmse))
metricas = pd.DataFrame(rows)
metricas.to_csv(os.path.join(OUT_DIR, "metricas.csv"), index=False)
metricas[(metricas.particao == "teste") & (metricas.participante == "AGREGADO")] \
    .pivot_table(index=["horizonte_min", "n_refeicoes"], columns="modelo", values=["MAE", "RMSE"]).round(2)

In [ ]:
metricas[(metricas.particao == "teste") & (metricas.participante != "AGREGADO")] \
    .pivot_table(index=["participante", "n_refeicoes"], columns=["horizonte_min", "modelo"], values="MAE").round(2)

## 8. Gráfico: as três primeiras refeições do teste

Sem escolher as "boas": são simplesmente as três primeiras refeições elegíveis do teste (ordem por participante e horário). A linha é a CGM observada de −55 a +60 min; os pontos são as previsões em +30 e +60 min.

In [ ]:
tst = pred[pred.particao == "teste"].sort_values(["subject", "origin_idx"]).head(3)
fig, axes = plt.subplots(1, len(tst), figsize=(5 * len(tst), 4), squeeze=False)
for ax, (_, r) in zip(axes[0], tst.iterrows()):
    g = grids[r.subject]; i = int(r.origin_idx)
    seg = g["cgm"].iloc[i - 11:i + 13]
    ax.plot((np.arange(-11, 13) * 5), seg.values, "k-o", ms=3, label="CGM observada")
    ax.axvline(0, color="gray", ls=":")
    for m, mk in [("persistencia", "s"), ("lstm_cgm", "^"), ("lstm_cgm_ctx", "D")]:
        ax.plot([30, 60], [r[f"{m}_30"], r[f"{m}_60"]], mk, label=m)
    ax.set_title(f"Subject {r.subject} — {r.t:%d/%m %H:%M}"); ax.set_xlabel("min desde a origem"); ax.set_ylabel("mg/dL")
axes[0][0].legend(fontsize=7); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "exemplos.png"), dpi=130); plt.show()

## 9. Salvar e responder: "o contexto melhorou a previsão nas refeições?"

In [ ]:
pred.rename(columns={"subject": "participante", "t": "origem", "g0": "glicemia_atual"}) \
    .to_csv(os.path.join(OUT_DIR, "previsoes.csv"), index=False)
torch.save(net_cgm.state_dict(), os.path.join(OUT_DIR, "pesos_lstm_cgm.pt"))
torch.save(net_ctx.state_dict(), os.path.join(OUT_DIR, "pesos_lstm_cgm_ctx.pt"))
joblib.dump({"cgm_mu": cgm_mu, "cgm_sd": cgm_sd, "ctx": ctx_sc, "y": y_sc}, os.path.join(OUT_DIR, "escaladores.joblib"))
json.dump(cut_times, open(os.path.join(OUT_DIR, "cortes_temporais.json"), "w"), indent=1, ensure_ascii=False)
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "torch": torch.__version__}
json.dump(dict(QUICK=QUICK, SEED=SEED, participantes=eligible, excluidos=discarded, max_epocas=MAX_EPOCHS,
               epocas_cgm=len(h_cgm), epocas_ctx=len(h_ctx), versoes=versions),
          open(os.path.join(OUT_DIR, "config.json"), "w"), indent=1, ensure_ascii=False)

a = metricas[(metricas.particao == "teste") & (metricas.participante == "AGREGADO")]
linhas = []
for h in [30, 60]:
    s = a[a.horizonte_min == h].set_index("modelo")
    dif = s.loc["lstm_cgm", "MAE"] - s.loc["lstm_cgm_ctx", "MAE"]
    linhas.append(f"+{h} min: contexto {'reduziu' if dif > 0 else 'NÃO reduziu'} o MAE em {abs(dif):.2f} mg/dL "
                  f"(LSTM CGM {s.loc['lstm_cgm','MAE']:.2f} × com contexto {s.loc['lstm_cgm_ctx','MAE']:.2f} × persistência {s.loc['persistencia','MAE']:.2f}).")
concl = "\n".join("- " + l for l in linhas)
print(concl)

resumo = f"""# Resumo — Protótipo 2 (LSTM por refeição)

- Modo: {'rápido' if QUICK else 'completo'} | participantes: {eligible} | excluídos: {discarded}
- Refeições por partição:\n\n{md_table(counts.reset_index().rename(columns={'index': 'participante'}), 0)}
- Tempos: treino {t_train:.1f}s, inferência {t_infer:.1f}s | épocas usadas: CGM {len(h_cgm)}, contexto {len(h_ctx)}

## Teste (agregado)

{md_table(a[['modelo','horizonte_min','n_refeicoes','MAE','RMSE']])}

## O contexto melhorou a previsão nas refeições?

{concl}

Limitações: o alvo inclui intervenções futuras desconhecidas na origem; disparo com resolução de 5 min;
zero = sem registro; prevê o futuro observado, não o efeito causal de uma dose.
"""
open(os.path.join(OUT_DIR, "resumo.md"), "w").write(resumo)
print(sorted(os.listdir(OUT_DIR)))